# Demo: построение карусели i2i (Т—Ж)

Ноутбук поднимает inference из serve_carousel.py и строит карусели K=12 для нескольких реальных статей-источников.

**Полный пайплайн:**

1. Загружаем каталог, эмбеддинги, coview-индекс и LightGBM-модель.
2. Для каждого source_id вызываем serve_one(ctx, source_id):
    - retrieval similar: kNN top-100 по cosine + coview top-50
    - retrieval explore: per-dept + cross-dept top-N (quality + trend)
    - LTR scoring: LightGBM LambdaRank, 17 фичей
    - reranking: дедуп, лимиты, softmax-семплинг explore, интерливинг 9 + 3
3. Печатаем результат: ранг, заголовок, рубрика, ltr_score, scoring_mode.

**Подготовка (один раз перед запуском):**

1. Локально собрать бандл командой:  python scripts/prepare_colab_bundle.py
2. Залить tj-recs-bundle.zip в Google Drive (любая папка).
3. В ячейке ниже указать путь к zip-файлу в BUNDLE_PATH.

Полный прогон: 2–3 минуты (большая часть — чтение эмбеддингов 400 МБ).

## 1. Установка зависимостей

In [ ]:
!pip install -q 'lightgbm>=4.3.0' 'catboost>=1.2.5' 'pyarrow>=14.0'
print('deps installed')

## 2. Монтируем Drive

Первый запуск спросит разрешение на доступ к Google Drive.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    print('Не в Colab — пропускаем mount.')

## 3. Распаковываем бандл

Если путь BUNDLE_PATH не подходит, поправь его. Распаковка занимает ~30 секунд.

In [ ]:
import os, zipfile, pathlib, time

BUNDLE_PATH = '/content/drive/MyDrive/Colab Notebooks/tj-recs-bundle.zip'
TARGET = pathlib.Path('/content/tj-recs')

if IN_COLAB:
    assert pathlib.Path(BUNDLE_PATH).exists(), f'Не нашёл бандл по пути {BUNDLE_PATH}'
    if not TARGET.exists():
        t = time.time()
        print(f'распаковываю {BUNDLE_PATH} -> {TARGET} ...')
        with zipfile.ZipFile(BUNDLE_PATH) as zf:
            zf.extractall('/content')
        print(f'   done in {time.time()-t:.1f}s')
    else:
        print(f'{TARGET} уже распакован')
    os.chdir(TARGET)
else:
    TARGET = pathlib.Path.cwd()
    while TARGET != TARGET.parent and not (TARGET / 'serve_carousel.py').exists():
        TARGET = TARGET.parent
    assert (TARGET / 'serve_carousel.py').exists(), 'не нашёл корень репо с serve_carousel.py'
    os.chdir(TARGET)

print('cwd:', os.getcwd())
print('files:', sorted(p.name for p in TARGET.iterdir() if not p.name.startswith('.'))[:20])

## 4. Поднимаем inference-контекст

Загружаем каталог, эмбеддинги, coview-индекс и LTR-модель. Это ~30–60 секунд (один раз на сессию).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from serve_carousel import build_context, serve_one

ctx = build_context()
print()
print('articles loaded:', f'{ctx.art.df.shape[0]:,}')
print('embeddings dim: ', ctx.art.X.shape[1])
print('coview sources: ', f'{len(ctx.coview_neigh_idx):,}')
print('LTR model:      ', type(ctx.model).__name__ if ctx.model else 'None (fallback heuristic)')

## 5. Карусель для одной статьи

Берём популярную статью и строим карусель K=12. Колонка scoring_mode показывает, использовалась ли LTR (ltr) или fallback (heuristic).

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 160)

SOURCE_ID = '56eadad6-91e6-46bc-b889-ad52194d9bf0'  # Что могут сделать судебные приставы

src_row = ctx.art.df[ctx.art.df['article_id'] == SOURCE_ID].iloc[0]
print('источник:')
print('  title:     ', src_row.get('article_base__title'))
print('  department:', src_row.get('article_base__department'))
print('  rubric:    ', src_row.get('article_base__rubric'))
print()

out = serve_one(ctx, SOURCE_ID)
cols = ['rank', 'candidate_article_id', 'candidate_title', 'candidate_rubric', 'mix', 'similarity', 'score', 'scoring_mode']
out[cols].rename(columns={
    'candidate_title': 'title',
    'candidate_rubric': 'rubric',
    'candidate_article_id': 'cand_id',
}).head(12)

## 6. Пачка карусели для разных источников

Показывает, что модель работает на разных типах контента (право, финансы, кино, медицина, кулинария…).

In [ ]:
SOURCES = [
    ('56eadad6-91e6-46bc-b889-ad52194d9bf0', 'судебные приставы'),
    ('c99f374c-6dff-46b8-8892-1804ea5b3ee4', 'лучшие фильмы 2024'),
    ('4c076836-dede-4dce-81b3-60afefc50c71', 'военная ипотека'),
    ('be91e9bc-e4e6-485c-9ece-770bdd5279e4', 'компенсация за отпуск'),
    ('364718f3-44bb-4d07-89d9-bdef067563c4', 'холестерин и яйца'),
    ('a3bbe8a8-f65b-4af6-bc6d-682662b96b84', 'как выбрать кондиционер'),
]

for sid, label in SOURCES:
    print('=' * 100)
    print(f'SOURCE [{label}]   id={sid}')
    print('=' * 100)
    try:
        out = serve_one(ctx, sid)
    except KeyError as e:
        print(f'  skip: {e}')
        continue
    short = out[['rank', 'candidate_title', 'candidate_rubric', 'mix', 'similarity', 'score', 'scoring_mode']]
    short = short.rename(columns={'candidate_title': 'title', 'candidate_rubric': 'rubric'})
    print(short.to_string(index=False))
    print()

## 7. (опционально) Карусель для случайной статьи

Если попросят показать на произвольной статье — запусти эту ячейку несколько раз.

In [ ]:
import random

views = pd.to_numeric(ctx.art.df.get('article_stats__stats_views', 0), errors='coerce').fillna(0)
df_with_views = ctx.art.df[views > 1000]
sid = random.choice(df_with_views['article_id'].tolist())
src = ctx.art.df[ctx.art.df['article_id'] == sid].iloc[0]
print('source:', src.get('article_base__title'), '  (dept=', src.get('article_base__department'), ')')
print()
out = serve_one(ctx, sid)
out[['rank', 'candidate_title', 'candidate_rubric', 'mix', 'similarity', 'score', 'scoring_mode']].head(12)

## 8. (опционально) Сравнение LTR vs fallback-эвристика

Выключим LTR и пересоберём ту же карусель эвристикой sim + priors. Видно, что именно дала LTR: перестановки в топе по ltr_score.

In [ ]:
saved_model = ctx.model
ctx.model = None  # отключаем LTR
out_heur = serve_one(ctx, SOURCE_ID)
ctx.model = saved_model  # возвращаем обратно

print('--- эвристический скор (sim + priors), scoring_mode = heuristic ---')
print(out_heur[['rank', 'candidate_title', 'mix', 'similarity', 'score', 'scoring_mode']].head(12).to_string(index=False))
print()
print('--- LTR (LightGBM LambdaRank), scoring_mode = ltr ---')
out_ltr = serve_one(ctx, SOURCE_ID)
print(out_ltr[['rank', 'candidate_title', 'mix', 'similarity', 'score', 'scoring_mode']].head(12).to_string(index=False))

## Что показывать на защите

**Минимум** (2 минуты): ячейки 1 → 2 → 3 → 4 → 5. Получаем готовую карусель из 12 статей.

**Если есть время:** ячейка 6 (пачка из 6 источников), 8 (сравнение с эвристикой).

**Если что-то отвалится** — открой serve_carousel.ipynb в соседней вкладке: там пошагово видна вся логика retrieval / scoring / reranking.